# MXfold2
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
method_name = "MXfold2"
base = Path.cwd()

In [ ]:
# Set installation directory
install_dir = "../tools"
os.makedirs(install_dir, exist_ok=True)
os.chdir(install_dir)

In [ ]:
mxfold2_env = "./mxfold2"
if not os.path.exists(mxfold2_env):
    print("Creating conda environment...")
    !conda create --prefix ./mxfold2 python=3.8 -y
    print("Conda environment created successfully")
else:
    print(f"Conda environment already exists at {mxfold2_env}")

In [ ]:
wheel_file = "mxfold2-0.1.1-cp38-cp38-linux_x86_64.whl"
if not os.path.exists(wheel_file):
    !wget -q https://github.com/keio-bioinformatics/mxfold2/releases/download/v0.1.1/mxfold2-0.1.1-cp38-cp38-linux_x86_64.whl
    print("MXfold2 wheel downloaded")
else:
    print(f"MXfold2 wheel already exists: {wheel_file}")

In [ ]:
mxfold2_env = "../tools/mxfold2"
wheel_file = "../tools/mxfold2-0.1.1-cp38-cp38-linux_x86_64.whl"

# Check if MXfold2 is already installed
try:
    result = subprocess.run(f"conda run -p {mxfold2_env} mxfold2 --version", shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print("MXfold2 is already installed")
    else:
        print("Installing MXfold2...")
        !conda run --no-capture-output -p {mxfold2_env} \
          python -m pip install -vv --progress-bar on {wheel_file}
        print("MXfold2 installed successfully")
except Exception as e:
    print(f"Installing MXfold2 due to error: {e}")
    !conda run --no-capture-output -p {mxfold2_env} \
      python -m pip install -vv --progress-bar on {wheel_file}
    print("MXfold2 installed successfully")

In [ ]:
# Change back to methods directory
os.chdir("../methods")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
gpu_id = 0  # Set to -1 to force CPU mode
mxfold2_env = "../tools/mxfold2"
use_gpu = gpu_id >= 0


In [ ]:
def run_folding(fasta_name):

    cmd = ["conda", "run", "-p", mxfold2_env, "mxfold2", "predict"]
    if use_gpu:
        cmd.extend(["--gpu", str(gpu_id)])
    cmd.append(fasta_name)
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error running MXfold2: {result.stderr}")
        return None
    
    # Write output to tmp.dot
    with open("tmp.dot", "w") as f:
        f.write(result.stdout)
    
    return "tmp.dot"

In [ ]:
out_dir = Path.cwd().parent / 'prediction'
os.makedirs(out_dir, exist_ok=True)

out_fasta_path = out_dir / (method_name + ".fasta")

if os.path.exists(out_fasta_path):
    os.remove(out_fasta_path)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}") 
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)
    
    # Write a one-sequence fasta
    tmp_fasta = f'MXfold2_tmp_{vid}.fasta'
    with open(tmp_fasta, 'w') as ofile:
        ofile.write(f'>{vid}\n{seq}\n')
    
    dot_file_name = run_folding(tmp_fasta)
    
    if dot_file_name:
        # Concatenate outputs
        os.system('cat ' + dot_file_name + ' >> ' + str(out_fasta_path))
    
    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s")
    
    # Clean up temporary files
    if os.path.exists(tmp_fasta):
        os.remove(tmp_fasta)
    if dot_file_name and os.path.exists(dot_file_name):
        os.remove(dot_file_name)

print(f"\nProcessing complete. Results saved to {out_fasta_path}")